# Redis + MySQL 会话存储整体架构

## 1. 启动与组件装配

`03_Memory/main.go` 提供 Redis 和 Redis + MySQL 两种示例。选择 Redis + MySQL 后，`RedisMyssqlStore` 按顺序初始化天气工具、`ChatStore`、ChatModel、摘要中间件和 `ChatModelAgent`，然后进入命令行对话循环。

```text
main.go
  └─ 选择 Redis + MySQL
      └─ RedisMyssqlStore
          ├─ 创建 get_weather 工具
          ├─ 初始化 ChatStore（MySQL + Redis）
          ├─ 创建 ChatModel 和摘要 Middleware
          ├─ 创建 ChatModelAgent（注册天气工具和 Middleware）
          └─ 循环读取用户输入并调用 RunChat
```

## 2. 每轮对话数据流

```text
用户输入
  │
  ▼
RunChat
  ├─ GetRecentMessages(sessionID)
  │    ├─ MySQL 查询最新摘要
  │    ├─ Redis 命中：按时间顺序读取最近消息
  │    └─ Redis 未命中：MySQL 读取最近 10 条 → 按时间正序恢复 → 回填 Redis
  │         两条路径都会将最新摘要作为 system 消息放在历史开头
  ├─ conversationHistory：过滤 tool 消息和带 tool_calls 的 assistant 中间消息
  ├─ 追加当前 user 消息
  ├─ SaveMessage：写入 MySQL 全量记录，并 RPush 到 Redis 最近消息列表
  ├─ Agent.Run(history)
  │    ├─ 按需调用 get_weather，工具调用和工具结果在本轮 Agent 内部处理
  │    ├─ 摘要 Middleware 在上下文消息达到 25 条时触发
  │    └─ 摘要回调将摘要保存到 MySQL
  ├─ 只保存不含 tool_calls 的最终 assistant 答复
  └─ 返回最终答复给终端

用户输入 ──► RunChat ──► Agent / get_weather ──► 最终答复
                 │                                  │
                 ├── ChatMessage ──► MySQL          ├── ChatMessage ──► MySQL
                 └── RPush ────────► Redis          └── RPush ────────► Redis
```

输入 `quit` 或标准输入结束后，程序从 MySQL 分页读取消息历史，并查询、显示最近一次已保存的摘要；函数返回时通过 `defer store.Close()` 关闭 MySQL 连接池和 Redis 客户端。

## 3. 存储职责

| 存储 | Key / 表 | 用途 | 顺序与保留策略 |
| --- | --- | --- | --- |
| Redis | `chat:memory:<sessionID>` | 缓存近期会话消息 | `RPush` 按时间正序写入，`LRANGE` 读取时无需反转；保留最近 10 条，TTL 为 24 小时 |
| MySQL | `chat_messages` | 保存完整消息历史 | 按自增 ID 持久化，支持分页读取 |
| MySQL | `chat_summaries` | 保存对话摘要及覆盖消息数 | 读取最新摘要；摘要 Middleware 触发时写入 |

`SaveMessage` 先写 MySQL，再通过 Redis Pipeline 执行 `RPush`、`LTrim` 和 `Expire`。Redis 缓存未命中时，`GetRecentMessages` 从 MySQL 取最近消息并回填缓存。

## 4. 当前实现注意事项

- `GetRecentMessages` 先读取最新摘要；无论 Redis 命中还是 MySQL 回源，都会将摘要放到 Agent 历史上下文的最前面。
- `NewChatStore` 自动迁移 `ChatMessage` 和 `ChatSummary` 表，因此首次运行会创建消息表与 `chat_summaries` 摘要表。
- 过滤 tool 消息和 assistant 工具调用中间态，是为了避免将不完整的工具调用序列带入下一轮模型请求；会话持久化的 assistant 内容是最终答复。

## 5. 关键代码

- `03_Memory/main.go`：示例菜单入口。
- `03_Memory/MemoryDemo/redis+mysql存储会话.go`：工具、摘要 Middleware、Agent 和 `RunChat` 流程。
- `pkg/chat_store.go`：MySQL 消息/摘要持久化，以及 Redis 最近消息缓存。
